# Getting Started with DNN

Before we move to harder tasks we will get familiar with Tensorflow and Keras. We will take a look at a first example of a neural network, which makes use of the Python library Keras to learn to classify hand-written digits. The problem we are trying to solve here is to classify grayscale images of handwritten digits (28 pixels by 28 pixels), into their 10 categories (0 to 9). The dataset we will use is the MNIST dataset. It's a set of 60,000 training images, plus 10,000 test images, assembled by the National Institute of Standards and Technology (the NIST in MNIST) in the 1980s. You can think of “solving” MNIST as the “Hello World” of deep learning – it's what you do to verify that your algorithms are working as expected.

Firstly, we will import Keras (based on the Python/Tensorflow version):


In [ ]:
#import keras  # - also works
#from tensorflow import keras # - recomended if you want the whole library
#from tensorflow.keras import XXX # - recomended if you want only a few functions

 The MNIST dataset comes pre-loaded in Keras, in the form of a set of four Numpy arrays:

In [ ]:
from keras.datasets import mnist

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

**DNN workflow**

*   Build the neural network architecture.
*   Train our neural network with the training data, train_images and train_labels. The network will then learn to associate images and labels.
*   The network will produce predictions for test_images, and we will verify if these predictions match the labels from test_labels.


**DNN architecture**

*   Our network will consist of a sequence of two Dense layers, which are densely-connected (also called “fully-connected”) neural layers.
*   The second (and last) layer is a 10-way “softmax” layer, which means it will return an array of 10 probability scores (summing to 1). Each score will be the probability that the current digit image belongs to one of our 10 digit classes.

**DNN training**

To make our network ready for training, we need to pick three more things, as part of the “compilation” step:

*   A loss function: This is how the network will be able to measure how good a job it is doing on its training data, and thus how it will be able to steer itself in the right direction.
*   An optimizer: this is the mechanism through which the network will update itself based on the data it sees and its loss function.
*   Metrics: to monitor during training and testing. Here we will only care about accuracy (the fraction of the images that were correctly classified).


Today, during our 3 examples we will use only the Sequential class. During our next meetings, I will introduce the functional API where we will be able to manipulate the data tensors that the model processes and apply layers to this tensors as if they were functions.

# Network Architecture

In [ ]:
from keras import models
from keras import layers

network = models.Sequential()
network.add(layers.Dense(512, activation='relu', input_shape=(28 * 28,))) # tutaj jest płaska zależnośc 28*28 (bo 28 pixeli) w konwolucyjnym bedzie przestrzebba 28 na 28
network.add(layers.Dense(10, activation='softmax')) #10 - tyle ile klas wyjściu

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


# Network Training

Once the model architecture is defined, the learning process can be configured in the compilation step. We specify the optimizer, loss function, and metrics.

In [ ]:
network.compile(optimizer='rmsprop',
                loss='categorical_crossentropy',
                metrics=['accuracy'])

**Data preparation**

Before training, we will preprocess our data by reshaping it into the shape that the network expects, and scaling it so that all values are in the [0, 1] interval. Our training images are being stored in an array of shape (60000, 28, 28) of type uint8 with values in the [0, 255] interval. Please transform it into a float32 array of shape (60000, 28 * 28) with values between 0 and 1. Necessary functions: reshape and astype. Perform it for both train and test examples.

Because of we use categorical_crossentropy loss function we need to convert data format:

In [ ]:
train_images = train_images.reshape((60000, 28 * 28))
train_images = train_images.astype('float32') / 255

test_images = test_images.reshape((10000, 28 * 28))
test_images = test_images.astype('float32') / 255

from keras.utils import to_categorical

train_labels = to_categorical(train_labels) # zmieniamy etykiety na inny format danych
test_labels = to_categorical(test_labels)

**Fit the model**

To train the network we call the fit method of the network with parameters epochs and batch_size. Set epochs to 5, and batch_size to 128.

In [ ]:
# fit the model
model = network.fit(train_images, train_labels, epochs=5, batch_size=128)

Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9246 - loss: 0.2618
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9681 - loss: 0.1080
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9787 - loss: 0.0710
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9850 - loss: 0.0509
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9883 - loss: 0.0390


# Network evaluation

Two quantities are being displayed during training: the “loss” of the network over the training data, and the accuracy of the network over the training data. We quickly reach an accuracy of 0.989 (i.e. 98.9%) on the training data. Now let's check that our model performs well on the test set too:

In [ ]:
test_loss, test_acc = network.evaluate(test_images, test_labels)

print('test_acc:', test_acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9806 - loss: 0.0594
test_acc: 0.9805999994277954


Our test set accuracy turns out to be 97.8% – that's quite a bit lower than the training set accuracy. This gap between training accuracy and test accuracy is an example of “overfitting”, the fact that machine learning models tend to perform worse on new data than on their training data.

# Convnet architecture

A basic convnet architecture is a a stack of Conv2D and MaxPooling2D layers followed by the classification part.

A convnet takes as input tensors of shape (image_height, image_width, image_channels) (not including the batch dimension). In our case, we will configure our convnet to process inputs of size (28, 28, 1), which is the format of MNIST images.

Pass the argument input_shape=(28, 28, 1) to our first layer
Number of channels is controlled by the first argument passed to each Conv2D layer (here 32 or 64).
The filters should have size 3×3.

Layers:

In [ ]:
#Feature extraction part
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1))) #32. - filtry konwolucyjne
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
#...
#...

#Classification part
model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(10, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Only the last layer changes the activation function to softmax.

Model Summary:


In [ ]:
_________________________________________________________________
Layer (type)                 Output Shape              Param #
=================================================================
conv2d_1 (Conv2D)            (None, 26, 26, 32)        320
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 13, 13, 32)        0
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 11, 11, 64)        18496
_________________________________________________________________
max_pooling2d_2 (MaxPooling2 (None, 5, 5, 64)          0
_________________________________________________________________
conv2d_3 (Conv2D)            (None, 3, 3, 64)          36928
_________________________________________________________________
flatten_1 (Flatten)          (None, 576)               0
_________________________________________________________________
dense_1 (Dense)              (None, 64)                36928
_________________________________________________________________
dense_2 (Dense)              (None, 10)                650
=================================================================
Total params: 93,322
Trainable params: 93,322
Non-trainable params: 0
____________________________

SyntaxError: invalid syntax (684725771.py, line 2)

Train the model once again

In [ ]:
model.compile(optimizer='rmsprop',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.fit(train_images, train_labels, epochs=5, batch_size=64)

Epoch 1/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9990 - loss: 0.0027
Epoch 2/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9991 - loss: 0.0028
Epoch 3/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9993 - loss: 0.0019
Epoch 4/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9995 - loss: 0.0021
Epoch 5/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9995 - loss: 0.0017    
